<a href="https://colab.research.google.com/github/telyotarsyn/simple_llm_rag/blob/main/Simple_RAG_petproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!nvidia-smi

Mon Sep 14 20:09:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             26W /   70W |     607MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Download PDF file
import os
import requests

# Get PDF document
pdf_path = "dune.pdf"

# Download PDF if it doesn't already exist
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  # The URL of the PDF you want to download
  url = "https://www.hetako.ee/kamp/ulme/Frank%20Herbert%20-%20Dune%201%20-%20Dune.pdf"

  # The local filename to save the downloaded file
  filename = pdf_path

  # Send a GET request to the URL
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Open a file in binary write mode and save the content to it
      with open(filename, "wb") as file:
          file.write(response.content)
      print(f"The file has been downloaded and saved as {filename}")
  else:
      print(f"Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} exists.")

File dune.pdf exists.


In [6]:
# Perform Google Colab installs (if running in Google Colab)
import os

if "COLAB_GPU" in os.environ:
    print("[INFO] Running in Google Colab, installing requirements.")
    !pip install -U torch # requires torch 2.1.1+ (for efficient sdpa implementation)
    !pip install PyMuPDF # for reading PDFs with Python
    !pip install tqdm # for progress bars
    !pip install sentence-transformers # for embedding models
    !pip install accelerate # for quantization model loading
    !pip install bitsandbytes # for quantizing models (less storage space)

KeyboardInterrupt: 

In [7]:
# Requires !pip install PyMuPDF, see: https://github.com/pymupdf/pymupdf
import fitz # (pymupdf, found this is better than pypdf for our use case, note: licence is AGPL-3.0, keep that in mind if you want to use any code commercially)
from tqdm.auto import tqdm # for progress bars, requires !pip install tqdm

def text_formatter(text: str) -> str:
    """Performs minor formatting on text."""
    cleaned_text = text.replace("\n", " ").strip() # note: this might be different for each doc (best to experiment)

    # Other potential text formatting functions can go here
    return cleaned_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path: str) -> list[dict]:
    """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
    doc = fitz.open(pdf_path)  # open a document
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):  # iterate the document pages
        text = page.get_text()  # get plain text encoded as UTF-8
        text = text_formatter(text)
        pages_and_texts.append({"page_number": page_number - 41,  # adjust page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(". ")),
                                "page_token_count": len(text) / 4,  # 1 token = ~4 chars, see: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them
                                "text": text})
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 119,
  'page_word_count': 29,
  'page_sentence_count_raw': 1,
  'page_token_count': 29.75,
  'text': 'Converted to  Converted to  Converted to  Converted to “PDF PDF PDF PDF” by  ” by  ” by  ” by ->MKM< >MKM< >MKM< >MKM<-'},
 {'page_number': -40,
  'page_char_count': 2291,
  'page_word_count': 498,
  'page_sentence_count_raw': 24,
  'page_token_count': 572.75,
  'text': 'Dune  Frank Herbert    Copyright 1965    Book 1  DUNE    = = = = = =     A beginning is the time for taking the most delicate care that the balances are  correct. This every sister of the Bene Gesserit knows. To begin your study of  the life of Muad\'Dib, then, take care that you first place him in his time: born  in the 57th year of the Padishah Emperor, Shaddam IV. And take the most special  care that you locate Muad\'Dib in his place: the planet Arrakis. Do not be  deceived by the fact that he was born on Caladan and lived his first fifteen  years there. Arrakis, the planet

In [8]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 41,
  'page_char_count': 3536,
  'page_word_count': 700,
  'page_sentence_count_raw': 40,
  'page_token_count': 884.0,
  'text': 'And Kynes, returning the stare, found himself troubled by a fact he had  observed here: This Duke was concerned more over the men that he was over the  spice. He risked his own life and that of his son to save the men. He passed off  the loss of a spice crawler with a gesture. The threat to men\'s lives had him in  a rage. A leader such as that would command fanatic loyalty. He would be  difficult to defeat.      Against his own will and all previous judgments, Kynes admitted to himself:  I like this Duke.    = = = = = =    Greatness is a transitory experience. It is never consistent. It depends in part  upon the myth-making imagination of humankind. The person who experiences  greatness must have a feeling for the myth he is in. He must reflect what is  projected upon him. And he must have a strong sense of the sardonic. This is  what uncou

In [9]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,119,29,1,29.75,Converted to Converted to Converted to Conv...
1,-40,2291,498,24,572.75,Dune Frank Herbert Copyright 1965 Book ...
2,-39,3591,736,79,897.75,"""Sleep well, you sly little rascal,"" said the ..."
3,-38,3562,736,44,890.50,When dawn touched Paul's window sill with yell...
4,-37,3513,788,63,878.25,"Now, there was a man who appreciated the power..."


In [10]:
# Get stats
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,345.00,345.00,345.00,345.00,345.00
mean,131.00,3500.50,740.63,46.57,875.13
std,99.74,364.47,79.78,12.04,91.12
min,-41.00,0.00,1.00,1.00,0.00
25%,45.00,3406.00,727.00,40.00,851.50
50%,131.00,3536.00,754.00,46.00,884.00
75%,217.00,3668.00,777.00,51.00,917.00
max,303.00,4169.00,837.00,130.00,1042.25


In [11]:
from spacy.lang.en import English # see https://spacy.io/usage for install instructions

nlp = English()

# Add a sentencizer pipeline, see https://spacy.io/api/sentencizer/
nlp.add_pipe("sentencizer")

# Create a document instance as an example
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2

# Access the sentences of the document
list(doc.sents)

[This is a sentence., This another sentence.]

In [12]:
for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)

    # Make sure all sentences are strings
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Count the sentences
    item["page_sentence_count_spacy"] = len(item["sentences"])
    # Inspect an example
random.sample(pages_and_texts, k=1)

  0%|          | 0/345 [00:00<?, ?it/s]

[{'page_number': 110,
  'page_char_count': 3688,
  'page_word_count': 777,
  'page_sentence_count_raw': 63,
  'page_token_count': 922.0,
  'text': '"I\'m sure we can produce an emergency to draw off any unwanted observers,  Nefud."      "I understand, m\'Lord. That\'s when Kynes can have his accident."      "Both Kynes and Hawat will have accidents then, Nefud. But only Kynes will  have a real accident. It\'s Hawat I want. Yes. Ah, yes."      Nefud blinked, swallowed. He appeared about to ask a question, but remained  silent.      "Hawat will be given both food and drink," the Baron said. "Treated with  kindness, with sympathy. In his water you will administer the residual poison  developed by the late Piter de Vries. And you will see that the antidote becomes  a regular part of Hawat\'s diet from this point on . . . unless I say otherwise."       "The antidote, yes." Nefud shook his head. "But--"       "Don\'t be dense, Nefud. The Duke almost killed me with that poison-capsule  tooth.

In [13]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,345.00,345.00,345.00,345.00,345.00,345.00
mean,131.00,3500.50,740.63,46.57,875.13,56.37
std,99.74,364.47,79.78,12.04,91.12,12.27
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,45.00,3406.00,727.00,40.00,851.50,50.00
50%,131.00,3536.00,754.00,46.00,884.00,57.00
75%,217.00,3668.00,777.00,51.00,917.00,64.00
max,303.00,4169.00,837.00,130.00,1042.25,89.00


In [14]:
# Define split size to turn groups of sentences into chunks
num_sentence_chunk_size = 10

# Create a function that recursively splits a list into desired sizes
def split_list(input_list: list,
               slice_size: int) -> list[list[str]]:
    """
    Splits the input_list into sublists of size slice_size (or as close as possible).

    For example, a list of 17 sentences would be split into two lists of [[10], [7]]
    """
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

# Loop through pages and texts and split sentences into chunks
for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])
# Sample an example from the group (note: many samples have only 1 chunk as they have <=10 sentences total)
random.sample(pages_and_texts, k=1)

  0%|          | 0/345 [00:00<?, ?it/s]

[{'page_number': 248,
  'page_char_count': 3778,
  'page_word_count': 816,
  'page_sentence_count_raw': 59,
  'page_token_count': 944.5,
  'text': 'terror. Without knowing why, her whole being trembled at what she had seen -- a  region where a wind blew and sparks glared, where rings of light expanded and  contracted, where rows of tumescent white shapes flowed over and under and  around the lights, driven by darkness and a wind out of nowhere.      Presently, she opened her eyes, saw Paul staring up at her. He still held  her hand, but the terrible rapport was gone. She quieted her trembling. Paul  released her hand. It was as though some crutch had been removed. She staggered  up and back, would have fallen had not Chani jumped to support her.      "Reverend Mother!" Chani said. "What is wrong?"       "Tired," Jessica whispered. "So . . . tired."      "Here," Chani said. "Sit here." She helped Jessica to a cushion against the  wall.      The strong young arms felt so good to Jessica.

In [15]:
# Create a DataFrame to get stats
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,345.00,345.00,345.00,345.00,345.00,345.00,345.00
mean,131.00,3500.50,740.63,46.57,875.13,56.37,6.10
std,99.74,364.47,79.78,12.04,91.12,12.27,1.26
min,-41.00,0.00,1.00,1.00,0.00,0.00,0.00
25%,45.00,3406.00,727.00,40.00,851.50,50.00,5.00
50%,131.00,3536.00,754.00,46.00,884.00,57.00,6.00
75%,217.00,3668.00,777.00,51.00,917.00,64.00,7.00
max,303.00,4169.00,837.00,130.00,1042.25,89.00,9.00


In [16]:
import re

# Split each chunk into its own item
pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]

        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  ", " ").strip()
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk) # ".A" -> ". A" for any full-stop/capital letter combo
        chunk_dict["sentence_chunk"] = joined_sentence_chunk

        # Get stats about the chunk
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4 # 1 token = ~4 characters

        pages_and_chunks.append(chunk_dict)

# How many chunks do we have?
len(pages_and_chunks)

  0%|          | 0/345 [00:00<?, ?it/s]

2104

In [17]:
# View a random sample
random.sample(pages_and_chunks, k=1)

[{'page_number': 130,
  'sentence_chunk': 'The hissing approach spread across the night behind them. They turned their heads as they walked, saw the mound of the coursing worm.   "Keep moving," Paul whispered. "Don\'t look back."   A grating sound of fury exploded from the rock shadows they had left. It was a flailing avalanche of noise.   "Keep moving," Paul repeating.',
  'chunk_char_count': 328,
  'chunk_word_count': 60,
  'chunk_token_count': 82.0}]

In [18]:
# Get stats about our chunks
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,2104.00,2104.00,2104.00,2104.00
mean,124.70,554.31,102.60,138.58
std,97.51,230.28,40.62,57.57
min,-41.00,2.00,1.00,0.50
25%,41.00,413.00,78.00,103.25
50%,121.00,530.00,99.00,132.50
75%,209.25,682.25,125.00,170.56
max,302.00,2024.00,335.00,506.00


In [19]:
# Show random chunks with under 30 tokens in length
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count: {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

Chunk token count: 3.25 | Text: questioning."
Chunk token count: 16.0 | Text: She nodded, letting him know that they left with her permission.
Chunk token count: 13.0 | Text: El-sayals frequently bring moisture to ground level.
Chunk token count: 27.75 | Text: "My Lord wishes?"She kept her head bowed, eyes shielded.   He gestured. "Have these basins and towels removed."
Chunk token count: 9.5 | Text: He stopped the circling, straightened.


In [20]:
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -40,
  'sentence_chunk': 'Dune Frank Herbert  Copyright 1965  Book 1 DUNE  = = = = = =   A beginning is the time for taking the most delicate care that the balances are correct. This every sister of the Bene Gesserit knows. To begin your study of the life of Muad\'Dib, then, take care that you first place him in his time: born in the 57th year of the Padishah Emperor, Shaddam IV. And take the most special care that you locate Muad\'Dib in his place: the planet Arrakis. Do not be deceived by the fact that he was born on Caladan and lived his first fifteen years there. Arrakis, the planet known as Dune, is forever his place. -from "Manual of Muad\'Dib" by the Princess Irulan    In the week before their departure to Arrakis, when all the final scurrying about had reached a nearly unbearable frenzy, an old crone came to visit the mother of the boy, Paul.   It was a warm night at Castle Caladan, and the ancient pile of stone that had served the Atreides family as home for t

In [21]:
!pip install -q langchain-huggingface

In [2]:
import torch
import transformers
import langchain
import numpy as np

print(" Все ключевые библиотеки RAG успешно импортированы!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

 Все ключевые библиотеки RAG успешно импортированы!
PyTorch version: 2.6.0+cu124
NumPy version: 2.5.2
CUDA available: True
GPU device: Tesla T4


In [25]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 226.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 141.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 58.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 79.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 91.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 97.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 88.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 101.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 13.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/

In [22]:
!python -c "import torchvision; print(torchvision.__version__)"


0.21.0+cu124


In [30]:
!pip install --force-reinstall --no-input torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (28 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp313-cp313-linux_x86_64.whl.metadata (6.1 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 136.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 238.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 143.0 MB/s eta 0:00:00
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl (664.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 66.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 75.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 117.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 89.6 MB/s eta 0:

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

# Инициализация модели через обертку LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'device': device}
)

sentences = [
    "The Sentences Transformers library provides an easy and open-source way to create embeddings.",
    "Sentences can be embedded one by one or as a list of strings.",
    "Embeddings are one of the most powerful concepts in machine learning!",
    "Learn to use embeddings well and you'll be well on your way to being an AI engineer."
]

# Команда embed_documents() принимает весь список строк
embeddings = embedding_model.embed_documents(sentences)
embeddings_dict = dict(zip(sentences, embeddings))

for sentence, embedding in embeddings_dict.items():
    print("Sentence:", sentence)
    print("First 5 elements:", embedding[:5])
    print("")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Sentence: The Sentences Transformers library provides an easy and open-source way to create embeddings.
First 5 elements: [-0.020798111334443092, 0.03031647950410843, -0.02012181468307972, 0.06864837557077408, -0.025525527074933052]

Sentence: Sentences can be embedded one by one or as a list of strings.
First 5 elements: [0.04317181184887886, -0.05387009680271149, -0.03780446946620941, 0.042723555117845535, -0.023540936410427094]

Sentence: Embeddings are one of the most powerful concepts in machine learning!
First 5 elements: [-0.029861142858862877, -0.013752218335866928, -0.047540176659822464, 0.02721269056200981, 0.03400547802448273]

Sentence: Learn to use embeddings well and you'll be well on your way to being an AI engineer.
First 5 elements: [-0.02207307144999504, 0.02089507132768631, -0.060300540179014206, 0.008439459837973118, 0.043765101581811905]



In [24]:
single_sentence = "Yo! How cool are embeddings?"
single_embedding = embedding_model.embed_query(single_sentence)
print(f"Sentence: {single_sentence}")
print(f"Embedding:\n{single_embedding}")
import torch
print(f"Embedding size: {torch.tensor(single_embedding).shape}")

Sentence: Yo! How cool are embeddings?
Embedding:
[-0.019744746387004852, -0.004510872531682253, -0.004984834231436253, 0.06554446369409561, -0.009876725263893604, 0.027283549308776855, 0.036642588675022125, -0.003302215598523617, 0.008500793017446995, 0.008249524980783463, -0.022849710658192635, 0.04024302214384079, -0.05752000957727432, 0.06336924433708191, 0.04432078078389168, -0.04495074599981308, 0.01252842228859663, -0.025201207026839256, -0.035529252141714096, 0.012955871410667896, 0.008670234121382236, -0.019291769713163376, 0.0035563285928219557, 0.01895061321556568, -0.014712822623550892, -0.009398438036441803, 0.007641688920557499, 0.00962190143764019, -0.005989279132336378, -0.03901693597435951, -0.05478247255086899, -0.005674568936228752, 0.011164519935846329, 0.040806759148836136, 1.7631907667237101e-06, 0.009152970276772976, -0.008772614412009716, 0.023938270285725594, -0.023278431966900826, 0.08049995452165604, 0.031917672604322433, 0.005125950090587139, -0.014770842157

In [25]:
%%time

# Uncomment to see how long it takes to create embeddings on CPU
# # Make sure the model is on the CPU
# embedding_model.to("cpu")

# # Embed each chunk one by one
# for item in tqdm(pages_and_chunks_over_min_token_len):
#     item["embedding"] = embedding_model.encode(item["sentence_chunk"])

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 9.54 µs


In [28]:
%%time

# Send the model to the GPU (already configured during initialization in SOYZniusXg0u)
# embedding_model.to("cuda") # requires a GPU installed, for reference on my local machine, I'm using a NVIDIA RTX 4090

# Create embeddings one by one on the GPU
for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.embed_query(item["sentence_chunk"])

  0%|          | 0/2029 [00:00<?, ?it/s]

CPU times: user 33.1 s, sys: 208 ms, total: 33.3 s
Wall time: 35 s


In [29]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]

In [30]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={'device': device},
    encode_kwargs={'batch_size': 32}  # Configures batch size for embed_documents
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [31]:
%%time

# Embed all text chunks
raw_embeddings = embedding_model.embed_documents(text_chunks)

# Convert list of embeddings to PyTorch Tensor (equivalent to convert_to_tensor=True)
text_chunk_embeddings = torch.tensor(raw_embeddings, device=device)

text_chunk_embeddings

CPU times: user 19.6 s, sys: 70.7 ms, total: 19.7 s
Wall time: 19.3 s


tensor([[ 0.0479, -0.0243, -0.0130,  ..., -0.0217,  0.0279, -0.0196],
        [ 0.0086,  0.0291, -0.0065,  ...,  0.0695, -0.0112, -0.0391],
        [ 0.0898,  0.0149, -0.0232,  ...,  0.0350, -0.0089, -0.0236],
        ...,
        [ 0.0003, -0.0612, -0.0006,  ...,  0.0386,  0.0054,  0.0093],
        [ 0.0312, -0.0377,  0.0366,  ...,  0.0344, -0.0048, -0.0165],
        [-0.0121, -0.0633,  0.0146,  ...,  0.0535, -0.0086, -0.0019]],
       device='cuda:0')

In [32]:
# Save embeddings to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
embeddings_df_save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(embeddings_df_save_path, index=False)

In [33]:
# Import saved file and view
text_chunks_and_embedding_df_load = pd.read_csv(embeddings_df_save_path)
text_chunks_and_embedding_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-40,Dune Frank Herbert Copyright 1965 Book 1 DUN...,1382,272,345.50,"[0.04793761298060417, -0.024343056604266167, -..."
1,-40,The old woman was a witch shadow -- hair like ...,558,100,139.50,"[0.00860720593482256, 0.02914559096097946, -0...."
2,-40,"She chuckled. ""But royalty has need of slyness...",281,58,70.25,"[0.08982345461845398, 0.01486481074243784, -0...."
3,-39,"""Sleep well, you sly little rascal,"" said the ...",669,136,167.25,"[0.03215346112847328, 0.05946078151464462, 0.0..."
4,-39,Kwisatz Haderach. There had been so many thi...,932,165,233.00,"[0.04089375585317612, 0.001995265716686845, -0..."


In [37]:
import json
import torch
import pandas as pd

# 1. Запись в CSV с сериализацией через json.dumps
raw_embeddings = embedding_model.embed_documents(text_chunks)
df = pd.DataFrame(pages_and_chunks_over_min_token_len)
df["embedding"] = [json.dumps(emb) for emb in raw_embeddings]

csv_save_path = "text_chunks_and_embeddings_df.csv"
df.to_csv(csv_save_path, index=False)

# 2. Чтение и десериализация через json.loads
df_loaded = pd.read_csv(csv_save_path)
df_loaded["embedding"] = df_loaded["embedding"].apply(json.loads)

# 3. Преобразование в тензор PyTorch
embeddings = torch.tensor(df_loaded["embedding"].tolist(), dtype=torch.float32).to(device)
print(embeddings.shape)

torch.Size([2029, 768])


In [38]:
text_chunks_and_embedding_df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-40,Dune Frank Herbert Copyright 1965 Book 1 DUN...,1382,272,345.50,"[0.04793761298060417, -0.024343056604266167, -..."
1,-40,The old woman was a witch shadow -- hair like ...,558,100,139.50,"[0.00860720593482256, 0.02914559096097946, -0...."
2,-40,"She chuckled. ""But royalty has need of slyness...",281,58,70.25,"[0.08982345461845398, 0.01486481074243784, -0...."
3,-39,"""Sleep well, you sly little rascal,"" said the ...",669,136,167.25,"[0.03215346112847328, 0.05946078151464462, 0.0..."
4,-39,Kwisatz Haderach. There had been so many thi...,932,165,233.00,"[0.04089375585317612, 0.001995265716686845, -0..."


In [39]:
embeddings[0]

tensor([ 4.7938e-02, -2.4343e-02, -1.2992e-02,  2.2343e-02, -4.2528e-03,
         3.3806e-02,  3.1966e-02,  8.0231e-03,  4.2489e-02, -1.8211e-02,
         8.4046e-02, -5.4432e-02,  1.5471e-02, -5.5373e-02, -2.6275e-03,
         2.2406e-02,  3.4723e-02, -2.4773e-02, -2.6130e-02,  1.9207e-02,
        -2.7781e-02,  4.8479e-03,  2.7200e-02,  9.3811e-03,  1.3458e-01,
        -1.2687e-02,  5.5638e-03,  8.1027e-03,  1.0328e-02, -1.4904e-02,
        -1.2637e-02, -1.1670e-02,  4.2506e-02, -2.7154e-02,  2.5789e-06,
        -4.6621e-02, -1.6685e-02, -3.8757e-02,  2.2177e-02, -1.0823e-02,
         3.4653e-02,  6.7642e-03, -1.6383e-02, -1.9323e-02,  1.8612e-02,
        -5.0098e-02,  2.1515e-02,  3.4432e-02, -2.6360e-02,  1.3497e-02,
         9.3486e-03, -5.3976e-02, -3.3663e-02,  8.3128e-03,  1.6931e-02,
         5.1280e-02, -2.2205e-03,  2.7144e-02,  8.0904e-03,  2.8443e-02,
        -1.9645e-02,  1.4887e-02, -4.0106e-02, -5.3086e-03,  1.2802e-01,
        -4.3501e-02, -1.8083e-02,  1.4957e-02,  2.9